<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/gemma4_topo_cbp_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## setup

In [ ]:
!pip install -U bitsandbytes>=0.46.1 -q

In [ ]:
!nvidia-smi

Sat Nov 21 01:00:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt

## train

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import math
import hashlib
import random
import torch
from torch.optim import Optimizer
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    get_cosine_schedule_with_warmup,
)
from datasets import load_dataset

# =====================================================================
# 0. SEED
# =====================================================================
SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# 1. TOPO-2026 OPTIMIZER
# =====================================================================
SAFETY_CONSTANT = 0.85
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]


def _stochastic_round_(src_fp32, dst_bf16):
    bits = src_fp32.view(torch.int32)
    bits.add_(torch.randint_like(bits, 0, 1 << 16))
    bits.bitwise_and_(-65536)
    dst_bf16.copy_(src_fp32)


class TopoAdamW(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2, embedding_layer=None, prime_anchors=None,
                 lambda_euler=SAFETY_CONSTANT):
        if embedding_layer is None:
            raise ValueError("'embedding_layer' must be passed explicitly.")
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

        self.embedding_layer = embedding_layer
        self.lambda_euler = lambda_euler
        w = embedding_layer.weight.data
        self.valid_anchors = [i for i in (prime_anchors or PRIME_ANCHORS) if i < w.shape[0]]
        self.baseline = {i: w[i].detach().clone() for i in self.valid_anchors}
        self.certified_sha256_digest = self._digest()
        print(f"[TopoAdamW] Baseline locked. Digest: {self.certified_sha256_digest[:16]}")

    def _digest(self):
        w = self.embedding_layer.weight.data
        buf = torch.cat([w[i].flatten().float() for i in self.valid_anchors]).cpu().numpy().tobytes()
        return hashlib.sha256(buf).hexdigest()

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        emb_w = self.embedding_layer.weight
        if self.baseline[self.valid_anchors[0]].device != emb_w.device:
            self.baseline = {i: t.to(emb_w.device) for i, t in self.baseline.items()}
        if emb_w.grad is not None:
            for i in self.valid_anchors:
                emb_w.grad[i].zero_()

        for group in self.param_groups:
            lr, wd, eps = group["lr"], group["weight_decay"], group["eps"]
            beta1, beta2 = group["betas"]
            for p in group["params"]:
                if not p.requires_grad or p.grad is None:
                    continue
                state = self.state[p]
                if not state:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p, dtype=torch.bfloat16)
                    state["exp_avg_sq"] = torch.zeros_like(p, dtype=torch.bfloat16)
                    state["comp"] = torch.zeros_like(p, dtype=torch.bfloat16)
                state["step"] += 1
                t = state["step"]
                m_buf, v_buf, comp = state["exp_avg"], state["exp_avg_sq"], state["comp"]

                g = p.grad.float()
                m = m_buf.float().mul_(beta1).add_(g, alpha=1.0 - beta1)
                v = v_buf.float().mul_(beta2).addcmul_(g, g, value=1.0 - beta2)
                del g
                w = p.float()
                denom = (v.sqrt() / math.sqrt(1.0 - beta2 ** t)).add_(eps)
                y = m.div(denom).mul_(-lr / (1.0 - beta1 ** t))
                del denom
                y.add_(w, alpha=-lr * wd).add_(comp.float())
                _stochastic_round_(m, m_buf)
                _stochastic_round_(v, v_buf)
                del m, v
                total = w.add_(y)
                new = total.to(torch.bfloat16)
                comp.copy_(total.sub_(new.float()))
                if p.dtype == torch.bfloat16:
                    p.copy_(new)
                else:
                    p.copy_(new.to(p.dtype))
                del y, w, total, new

                if p is emb_w:
                    for i in self.valid_anchors:
                        p[i].copy_(self.baseline[i].to(device=p.device, dtype=p.dtype))
                        comp[i].zero_()
        return loss

    def audit_telemetry(self):
        w = self.embedding_layer.weight.data
        drift = max(torch.max(torch.abs(w[i].float() - self.baseline[i].to(w.device).float())).item()
                    for i in self.valid_anchors)
        h = self._digest()
        return drift, h, h == self.certified_sha256_digest


# =====================================================================
# 2. MODEL / TOKENIZER
# =====================================================================
model_id = "google/gemma-4-26B-A4B"

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    dtype=torch.bfloat16,
    device_map={"": 0} if torch.cuda.is_available() else "auto",
    low_cpu_mem_usage=True,
)
model.config.use_cache = False

for p in model.parameters():
    p.requires_grad = False
for name, mod in model.named_modules():
    if "lm_head" in name or "norm" in name:
        for p in mod.parameters(recurse=False):
            p.requires_grad = True

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"[*] Trainable parameters: {sum(p.numel() for p in trainable_params):,}")

topo_optimizer = TopoAdamW(
    trainable_params,
    lr=1e-4,
    weight_decay=1e-2,
    embedding_layer=model.get_input_embeddings(),
    prime_anchors=PRIME_ANCHORS,
)

# =====================================================================
# 3. DATA
# =====================================================================
def build_user_prompt(sample):
    instruction = (sample.get("instruction") or "").strip()
    extra_input = (sample.get("input") or "").strip()
    return f"{instruction}\n\nInput:\n{extra_input}" if extra_input else instruction


def preprocess_function(sample):
    user = build_user_prompt(sample)
    if getattr(tokenizer, "chat_template", None):
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True
        )
    else:
        bos = tokenizer.bos_token or ""
        prompt_text = f"{bos}{user}\n"

    eos_token = tokenizer.eos_token or ""
    answer_text = (sample.get("output") or "") + eos_token
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]
    input_ids = (list(prompt_ids) + answer_ids)[:512]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:512]
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}


raw = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=SEED)
train_ds = raw.select(range(12500)).map(preprocess_function, remove_columns=raw.column_names)
eval_ds = raw.select(range(12500, 13000)).map(preprocess_function, remove_columns=raw.column_names)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collator)
eval_loader = DataLoader(eval_ds, batch_size=2, shuffle=False, collate_fn=collator)

# =====================================================================
# 4. PURE PYTORCH TRAINING & AUDIT LOOP
# =====================================================================
epochs = 3
grad_accum = 16
max_grad_norm = 1.0
total_steps = (len(train_loader) // grad_accum) * epochs
scheduler = get_cosine_schedule_with_warmup(
    topo_optimizer, num_warmup_steps=20, num_training_steps=max(total_steps, 1)
)


@torch.no_grad()
def evaluate_loss():
    model.eval()
    losses = []
    for batch in eval_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            out = model(**batch)
        losses.append(out.loss.item())
    model.train()
    return sum(losses) / max(len(losses), 1)


print(f"[baseline] validation loss before training: {evaluate_loss():.4f}", flush=True)

output_dir = "/content/drive/MyDrive/topogpt/Gemma-4-26B-A4B-alpaca-topo-4bit-train"
os.makedirs(output_dir, exist_ok=True)

model.train()
topo_optimizer.zero_grad()
global_step = 0
running_loss = 0.0

for epoch in range(1, epochs + 1):
    for step_idx, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss = outputs.loss / grad_accum

        loss.backward()
        running_loss += outputs.loss.item()

        if step_idx % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
            topo_optimizer.step()
            scheduler.step()
            topo_optimizer.zero_grad()
            global_step += 1

            if global_step % 20 == 0:
                avg_loss = running_loss / (20 * grad_accum)
                lr_now = scheduler.get_last_lr()[0]
                print(f"step {global_step:>5} | train loss {avg_loss:.4f} | lr {lr_now:.2e}", flush=True)
                running_loss = 0.0

            if global_step % 100 == 0:
                val_loss = evaluate_loss()
                print(f"[eval] step {global_step:>5} | val loss {val_loss:.4f}", flush=True)

    epoch_dir = os.path.join(output_dir, f"epoch-{epoch}")
    os.makedirs(epoch_dir, exist_ok=True)
    model.save_pretrained(epoch_dir)
    tokenizer.save_pretrained(epoch_dir)
    print(f"[*] Saved epoch {epoch} checkpoint -> {epoch_dir}", flush=True)

max_drift, current_hash, hash_matched = topo_optimizer.audit_telemetry()
print(f"[TopoAudit] Max Anchor Drift: {max_drift:.6e} | Hash Matched: {hash_matched} | Digest: {current_hash[:16]}")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/103k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (15 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[*] Saved epoch 3 checkpoint -> /content/drive/MyDrive/topogpt/Gemma-4-26B-A4B-alpaca-topo-4bit-train/epoch-3
[TopoAudit] Max Anchor Drift: 0.000000e+00 | Hash Matched: True | Digest: c573d4a1c789de48


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/topogpt/Gemma-4-26B-A4B-alpaca-topo-4bit-train/tokenizer_config.json',
 '/content/drive/MyDrive/topogpt/Gemma-4-26B-A4B-alpaca-topo-4bit-train/tokenizer.json')

In [ ]:
!ls -ltha /content/drive/MyDrive/topogpt/Gemma-4-26B-A4B-alpaca-topo-4bit-train/

total 46G
drwx------  5 root root 4.0K Nov 21 03:31 .
-rw-------  1 root root  31M Nov 21 03:31 tokenizer.json
-rw-------  1 root root 1.5K Nov 21 03:31 tokenizer_config.json
-rw-------  1 root root  46G Nov 21 03:31 model.safetensors
-rw-------  1 root root 4.2K Nov 21 03:28 config.json
drwx------  2 root root 4.0K Nov 21 03:28 epoch-3
-rw-------  1 root root  177 Nov 21 03:28 generation_config.json
drwx------  2 root root 4.0K Nov 21 02:40 epoch-2
drwx------  2 root root 4.0K Nov 21 01:52 epoch-1
drwx------ 24 root root 4.0K Nov 21 01:05 ..


In [ ]:
%cd /content/drive/MyDrive/topogpt/Gemma-4-26B-A4B-alpaca-topo-4bit-train
!du -sh *

/content/drive/MyDrive/topogpt/Gemma-4-26B-A4B-alpaca-topo-4bit-train
4.5K	config.json
46G	epoch-1
46G	epoch-2
46G	epoch-3
512	generation_config.json
46G	model.safetensors
1.5K	tokenizer_config.json
31M	tokenizer.json


In [1]:
!nvidia-smi

Sat Nov 21 05:29:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   27C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import gc
import hashlib
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 123
EXPECTED_DIGEST = "c573d4a1c789de48"
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]

LOCAL_BASE_DIR = "/content/drive/MyDrive/topogpt/Gemma-4-26B-A4B-alpaca-topo-4bit-train"
DRIVE_FINAL_DIR= "/content/drive/MyDrive/topogpt/Gemma-4-26B-A4B-alpaca-topo-4bit"
FINAL_DIR = DRIVE_FINAL_DIR if os.path.isdir(DRIVE_FINAL_DIR) else LOCAL_BASE_DIR

CHECKPOINTS = [
    ("epoch-1", os.path.join(LOCAL_BASE_DIR, "epoch-1")),
    ("epoch-2", os.path.join(LOCAL_BASE_DIR, "epoch-2")),
    ("epoch-3", os.path.join(LOCAL_BASE_DIR, "epoch-3")),
    ("final", FINAL_DIR),
]

PROMPTS = [
    "Explain the difference between synchronous and asynchronous execution in two sentences.",
    "List three practical ways to reduce GPU memory usage during neural network training.",
    "Given a table orders(order_id, customer_id, total_amount, created_at), write a SQL query to find the top 5 customers by total spend.",
]


def compute_anchor_digest(model, anchors) -> str | None:
    emb = model.get_input_embeddings()
    if emb is None or not hasattr(emb, "weight"):
        return None
    w = emb.weight.detach()
    valid = [idx for idx in anchors if 0 <= idx < w.shape[0]]
    chunks = [
        w[idx].reshape(-1).to(dtype=torch.float32).cpu().numpy().tobytes()
        for idx in valid
    ]
    return hashlib.sha256(b"".join(chunks)).hexdigest()


def format_prompt(tokenizer, prompt_text: str) -> str:
    if getattr(tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": prompt_text}]
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    return f"Instruction:\n{prompt_text}\n\nResponse:\n"


def run_comparison():
    has_cuda = torch.cuda.is_available()
    results = {}

    for label, ckpt_path in CHECKPOINTS:
        if not os.path.isdir(ckpt_path):
            print(f"[!] Skipping {label}: directory not found at {ckpt_path}")
            continue

        torch.manual_seed(SEED)
        if has_cuda:
            torch.cuda.manual_seed_all(SEED)

        print(f"\n{'=' * 72}")
        print(f"[*] Loading checkpoint: {label} ({ckpt_path})")

        tokenizer = AutoTokenizer.from_pretrained(ckpt_path, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(
            ckpt_path,
            attn_implementation="sdpa",
            dtype=torch.bfloat16,
            device_map={"": 0} if has_cuda else "auto",
        )
        model.eval()

        digest = compute_anchor_digest(model, PRIME_ANCHORS)
        matched = (digest == EXPECTED_DIGEST) if digest else False
        print(f"[TopoAudit] {label} | Anchor Digest: {digest} | Matched: {matched}")

        ckpt_outputs = []
        pad_id = (
            tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id
        )

        for idx, p in enumerate(PROMPTS, start=1):
            formatted = format_prompt(tokenizer, p)
            inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
            input_len = inputs["input_ids"].shape[1]

            torch.manual_seed(SEED)
            with torch.no_grad():
                gen_ids = model.generate(
                    **inputs,
                    max_new_tokens=128,
                    do_sample=False,
                    pad_token_id=pad_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            completion_ids = gen_ids[0][input_len:]
            text = tokenizer.decode(completion_ids, skip_special_tokens=True).strip()
            ckpt_outputs.append(text)

            print(f"\n--- [{label}] Prompt {idx} ---")
            print(f"Q: {p}")
            print(f"A: {text}")

        results[label] = {
            "digest": digest,
            "matched": matched,
            "outputs": ckpt_outputs,
        }

        del model
        del tokenizer
        gc.collect()
        if has_cuda:
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    print(f"\n{'=' * 72}")
    print("[*] Summary Across All Four Checkpoints")
    for label, info in results.items():
        short_hash = info["digest"][:16] if info["digest"] else "N/A"
        print(f"  - {label:8s} | Digest: {short_hash} | Hash Matched: {info['matched']}")


if __name__ == "__main__":
    run_comparison()

In [ ]:
!ls -lh /content/drive/MyDrive/topogpt/Gemma-4-26B-A4B-alpaca-topo-4bit-train/
!df -h /

total 46G
-rw------- 1 root root 4.2K Nov 21 03:28 config.json
drwx------ 2 root root 4.0K Nov 21 01:50 epoch-1
drwx------ 2 root root 4.0K Nov 21 02:37 epoch-2
drwx------ 2 root root 4.0K Nov 21 03:25 epoch-3
-rw------- 1 root root  177 Nov 21 03:28 generation_config.json
-rw------- 1 root root  46G Nov 21 03:31 model.safetensors
-rw------- 1 root root 1.5K Nov 21 03:31 tokenizer_config.json
-rw------- 1 root root  31M Nov 21 03:31 tokenizer.json
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G  1.3G  235G   1% /


In [ ]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize

-rwx------ 1 root root 18K Nov  2 10:56 /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize


## gguf

In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
LOCAL_BASE_DIR = "/content/drive/MyDrive/topogpt/Gemma-4-26B-A4B-alpaca-topo-4bit-train"
DRIVE_BASE_DIR = "/content/drive/MyDrive/topogpt"
DRIVE_FINAL_DIR = os.path.join(DRIVE_BASE_DIR, "Gemma-4-26B-A4B-alpaca-topo-4bit")
DRIVE_GGUF_DIR = os.path.join(DRIVE_BASE_DIR, "gguf")

FINAL_DIR = DRIVE_FINAL_DIR if os.path.isdir(DRIVE_FINAL_DIR) else LOCAL_BASE_DIR

CHECKPOINTS = {
    "epoch-1": os.path.join(LOCAL_BASE_DIR, "epoch-1"),
    "final": FINAL_DIR,
}


def find_quantize_binary(llama_cpp_dir: str) -> str:
    candidates = [
        os.path.join(llama_cpp_dir, "build", "bin", "llama-quantize"),
        os.path.join(llama_cpp_dir, "llama-quantize"),
        os.path.join(llama_cpp_dir, "build", "bin", "quantize"),
        os.path.join(llama_cpp_dir, "quantize"),
    ]
    for path in candidates:
        if os.path.isfile(path) and os.access(path, os.X_OK):
            return path
    raise FileNotFoundError(
        f"Could not find executable llama-quantize in {llama_cpp_dir}/build/bin or {llama_cpp_dir}."
    )


def strip_quantization_config(model_dir: str):
    cfg_path = os.path.join(model_dir, "config.json")
    if not os.path.isfile(cfg_path):
        return
    with open(cfg_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)
    if "quantization_config" in cfg:
        del cfg["quantization_config"]
        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, indent=2)


def run_cmd(cmd: list[str], label: str):
    print(f"[*] {label}: {' '.join(cmd)}")
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout[-2500:])
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr[-2500:])
        raise RuntimeError(f"{label} failed with exit code {proc.returncode}")


def dequant_convert_and_quantize(
    checkpoint_label: str = "epoch-1",
    base_gguf_type: str = "f16",
    quant_type: str = "Q8_0",
    local_scratch_dir: str = "/content/gguf_work",
    drive_gguf_dir: str = DRIVE_GGUF_DIR,
    llama_cpp_dir: str = LLAMA_CPP_DIR,
):
    ckpt_dir = CHECKPOINTS.get(checkpoint_label)
    if not ckpt_dir or not os.path.isdir(ckpt_dir):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_dir}")

    convert_script = os.path.join(llama_cpp_dir, "convert_hf_to_gguf.py")
    if not os.path.isfile(convert_script):
        raise FileNotFoundError(f"Missing {convert_script}")

    quant_bin = find_quantize_binary(llama_cpp_dir)
    dequant_dir = os.path.join(local_scratch_dir, f"dequant_{checkpoint_label}_bf16")
    os.makedirs(dequant_dir, exist_ok=True)
    os.makedirs(drive_gguf_dir, exist_ok=True)

    print(f"[1/3] Dequantizing '{checkpoint_label}' ({ckpt_dir}) -> {dequant_dir}")
    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        ckpt_dir,
        attn_implementation="sdpa",
        dtype=torch.bfloat16,
        device_map="auto",
    )

    try:
        import bitsandbytes as bnb
        for module in model.modules():
            if isinstance(module, bnb.nn.Linear4bit) and getattr(module, "weight", None) is not None:
                if hasattr(module.weight, "quant_state") and module.weight.quant_state is not None:
                    dequant = bnb.functional.dequantize_4bit(
                        module.weight.data, module.weight.quant_state
                    ).to(torch.bfloat16)
                    module.weight = torch.nn.Parameter(dequant, requires_grad=False)
    except Exception as exc:
        print(f"[Warning] 4-bit unpack note: {exc}")

    model.save_pretrained(
        dequant_dir,
        safe_serialization=True,
        save_original_format=False,
    )
    tokenizer.save_pretrained(dequant_dir)
    strip_quantization_config(dequant_dir)

    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    base_gguf_path = os.path.join(
        drive_gguf_dir, f"Gemma-4-26B-A4B-alpaca-topo-{checkpoint_label}-{base_gguf_type}.gguf"
    )
    run_cmd(
        [
            sys.executable,
            convert_script,
            dequant_dir,
            "--outfile",
            base_gguf_path,
            "--outtype",
            base_gguf_type,
        ],
        label="Convert HF to base GGUF",
    )

    shutil.rmtree(dequant_dir, ignore_errors=True)

    final_gguf_path = os.path.join(
        drive_gguf_dir, f"Gemma-4-26B-A4B-alpaca-topo-{checkpoint_label}-{quant_type}.gguf"
    )
    run_cmd(
        [quant_bin, base_gguf_path, final_gguf_path, quant_type],
        label=f"Quantize GGUF to {quant_type}",
    )

    if os.path.isfile(final_gguf_path) and os.path.isfile(base_gguf_path):
        os.remove(base_gguf_path)

    size_gb = os.path.getsize(final_gguf_path) / (1024 ** 3)
    print(f"[+] Done! Final quantized GGUF on Drive: {final_gguf_path} ({size_gb:.2f} GB)")
    return final_gguf_path


if __name__ == "__main__":
    dequant_convert_and_quantize(
        checkpoint_label="epoch-1",
        base_gguf_type="f16",
        quant_type="Q8_0",
        local_scratch_dir="/content/gguf_work",
        drive_gguf_dir=DRIVE_GGUF_DIR,
        llama_cpp_dir=LLAMA_CPP_DIR,
    )

## gguf eval

In [2]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.6/893.6 MB 52.2 MB/s  0:00:16
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [ ]:
!ls -ltha  /content/drive/MyDrive/topogpt/Gemma-4-26B-A4B-alpaca-topo-4bit-train

total 46G
-rw-------  1 root root  31M Nov 21 03:31 tokenizer.json
-rw-------  1 root root 1.5K Nov 21 03:31 tokenizer_config.json
-rw-------  1 root root  46G Nov 21 03:31 model.safetensors
-rw-------  1 root root 4.2K Nov 21 03:28 config.json
-rw-------  1 root root  177 Nov 21 03:28 generation_config.json
drwx------  2 root root 4.0K Nov 21 03:25 epoch-3
drwx------  2 root root 4.0K Nov 21 02:37 epoch-2
drwx------  2 root root 4.0K Nov 21 01:50 epoch-1
drwx------  5 root root 4.0K Nov 21 01:05 .
drwx------ 24 root root 4.0K Oct  9 10:40 ..


In [ ]:
import gc
import torch
from llama_cpp import Llama

GGUF_PATH = "/content/drive/MyDrive/topogpt/gguf/Gemma-4-26B-A4B-alpaca-topo-epoch-1-f16.gguf"
SEED = 123

PROMPTS = [
    "Explain the difference between synchronous and asynchronous execution in two sentences.",
    "List three practical ways to reduce GPU memory usage during neural network training.",
    "Given a table orders(order_id, customer_id, total_amount, created_at), write a SQL query to find the top 5 customers by total spend.",
]


def format_alpaca(instruction: str) -> str:
    return (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{instruction}\n\n### Response:\n"
    )


llm = Llama(
    model_path=GGUF_PATH,
    n_gpu_layers=-1,
    n_ctx=2048,
    flash_attn=True,
    seed=SEED,
    verbose=False,
)


for idx, prompt in enumerate(PROMPTS, start=1):
    out = llm(
        format_alpaca(prompt),
        max_tokens=256,
        temperature=0.0,
        stop=["### Instruction:", "\n###"],
        seed=SEED,
        echo=False,
    )
    choice = out["choices"][0]
    text = choice["text"].strip()
    reason = choice.get("finish_reason")
    print(f"--- Prompt {idx} (finish_reason={reason}) ---\nQ: {prompt}\nA: {text}\n")

del llm
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


--- Prompt 1 (finish_reason=stop) ---
Q: Explain the difference between synchronous and asynchronous execution in two sentences.
A: Synchronous execution means that tasks are completed one after another, with each task waiting for the previous one to finish before starting. Asynchronous execution, on the other hand, allows tasks to be completed concurrently, with each task running independently and not waiting for the others to finish.

--- Prompt 2 (finish_reason=stop) ---
Q: List three practical ways to reduce GPU memory usage during neural network training.
A: 1. **Model Compression:** One of the most effective ways to reduce GPU memory usage is by compressing the model. This can be done through techniques such as pruning, quantization, and knowledge distillation. Pruning involves removing weights from the model that have little impact on the output, while quantization reduces the precision of the weights, often from 32-bit floating point to 8-bit integers. Knowledge distillation in

In [ ]:
!chmod a+x /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize
!ls -ltha /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize

-rwx------ 1 root root 18K Nov  2 10:56 /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize


In [ ]:
!ls /content/drive/MyDrive/topogpt/gguf/

Gemma-4-26B-A4B-alpaca-topo-epoch-1-f16.gguf  nemotron


In [ ]:
import os
import subprocess

QUANT_BIN_SRC = "/content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize"
LLAMA_BUILD_DIR = "/content/drive/MyDrive/topogpt/llama.cpp/build"
INPUT_GGUF = "/content/drive/MyDrive/topogpt/gguf/Gemma-4-26B-A4B-alpaca-topo-epoch-1-f16.gguf"
OUTPUT_GGUF = "/content/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf"
QUANT_TYPE = "Q8_0"

if not os.path.isfile(QUANT_BIN_SRC):
    raise FileNotFoundError(f"Not found: {QUANT_BIN_SRC}")
if not os.path.isfile(INPUT_GGUF):
    raise FileNotFoundError(f"Not found: {INPUT_GGUF}")

env = os.environ.copy()
lib_dirs = [
    os.path.join(LLAMA_BUILD_DIR, "bin"),
    os.path.join(LLAMA_BUILD_DIR, "src"),
    os.path.join(LLAMA_BUILD_DIR, "ggml", "src"),
]
existing_ld = env.get("LD_LIBRARY_PATH", "")
env["LD_LIBRARY_PATH"] = ":".join([d for d in lib_dirs if os.path.isdir(d)] + ([existing_ld] if existing_ld else []))

cmd = [QUANT_BIN_SRC, INPUT_GGUF, OUTPUT_GGUF, QUANT_TYPE]
print("[*] Running:", " ".join(cmd))
print("[*] LD_LIBRARY_PATH:", env["LD_LIBRARY_PATH"])
proc = subprocess.run(cmd, env=env, text=True, capture_output=True)
if proc.stdout:
    print(proc.stdout[-2500:])
if proc.returncode != 0:
    if proc.stderr:
        print(proc.stderr[-2500:])
    raise RuntimeError(f"llama-quantize failed with exit code {proc.returncode}")

size_gb = os.path.getsize(OUTPUT_GGUF) / (1024 ** 3)
print(f"[+] Done: {OUTPUT_GGUF} ({size_gb:.2f} GB)")

[*] Running: /content/drive/MyDrive/topogpt/llama.cpp/build/bin/llama-quantize /content/drive/MyDrive/topogpt/gguf/Gemma-4-26B-A4B-alpaca-topo-epoch-1-f16.gguf /content/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf Q8_0
[*] LD_LIBRARY_PATH: /content/drive/MyDrive/topogpt/llama.cpp/build/bin:/content/drive/MyDrive/topogpt/llama.cpp/build/src:/content/drive/MyDrive/topogpt/llama.cpp/build/ggml/src:/usr/lib64-nvidia

llama_quantize: quantize time = 75978.82 ms
llama_quantize:    total time = 75978.82 ms

[+] Done: /content/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf (25.02 GB)


In [1]:
import gc
import torch
from llama_cpp import Llama

GGUF_PATH = "/content/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf"
SEED = 123

PROMPTS = [
    "Explain the difference between synchronous and asynchronous execution in two sentences.",
    "List three practical ways to reduce GPU memory usage during neural network training.",
    "Given a table orders(order_id, customer_id, total_amount, created_at), write a SQL query to find the top 5 customers by total spend.",
]


def format_alpaca(instruction: str) -> str:
    return (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{instruction}\n\n### Response:\n"
    )


print(f"[*] Loading GGUF: {GGUF_PATH}")
llm = Llama(
    model_path=GGUF_PATH,
    n_gpu_layers=-1,
    n_ctx=2048,
    flash_attn=True,
    seed=SEED,
    verbose=False,
)
print(f"[+] Loaded GGUF: {GGUF_PATH}")


for idx, prompt in enumerate(PROMPTS, start=1):
    out = llm(
        format_alpaca(prompt),
        max_tokens=256,
        temperature=0.0,
        stop=["### Instruction:", "\n###"],
        seed=SEED,
        echo=False,
    )
    choice = out["choices"][0]
    text = choice["text"].strip()
    reason = choice.get("finish_reason")
    print(f"--- Prompt {idx} (finish_reason={reason}) ---\nQ: {prompt}\nA: {text}\n")

del llm
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

[*] Loading GGUF: /content/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


[+] Loaded GGUF: /content/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf
--- Prompt 1 (finish_reason=stop) ---
Q: Explain the difference between synchronous and asynchronous execution in two sentences.
A: Synchronous execution refers to a process where tasks are completed one after another, with each task waiting for the previous one to complete before starting. In contrast, asynchronous execution allows tasks to be started and completed independently, without waiting for other tasks to finish, enabling more efficient use of resources and faster overall execution.

--- Prompt 2 (finish_reason=stop) ---
Q: List three practical ways to reduce GPU memory usage during neural network training.
A: 1. **Model Compression:** One of the most effective ways to reduce GPU memory usage is by compressing the model. This can be done through techniques such as pruning, quantization, and knowledge distillation. Pruning involves removing weights from the model that have little impact on the output, whil

In [3]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/

total 308G
drwx------ 51 root root 4.0K Nov 21 04:41 ..
-rw-------  1 root root 6.6K Nov 20 20:05 somala_drive_gguf_audit.json
-rw-------  1 root root  27G Nov 14 20:14 qwen3.8-27b-topocbp-Q8_0.gguf
-rw-------  1 root root  11G Nov 12 06:55 deepseek_moe_16b_topocbp_q4_k_m.gguf
-rw-------  1 root root  70G Nov  9 06:01 deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf
-rw-------  1 root root  32G Nov  9 05:57 sarvam-30b-topo-governed-q8_0.gguf
-rw-------  1 root root  16G Nov  9 04:40 muse-glimmer-30b-card-faithful-Q4_K_M.gguf
-rw-------  1 root root  15G Nov  9 04:36 gpt-oss-20b-topo-governed-Q4_K_M.gguf
-rw-------  1 root root  15G Nov  9 04:34 certified-scorpion-topo-qwen-bf16.gguf
-rw-------  1 root root  40G Nov  9 04:31 llama-3.1-70b-topo-governed-q4_k_m.gguf
-rw-------  1 root root 5.0G Nov  9 04:23 nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf
-rw-------  1 root root  80G Nov  9 02:27 mixtral_8x22b_q4_k_m.gguf
drwx------  2 root root 4.0K Nov  9 02:15 .


In [4]:
!mv /content/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf /content/drive/MyDrive/GGUF_Models/

In [5]:
!ls -ltha /content/drive/MyDrive/GGUF_Models/

total 333G
drwx------  2 root root 4.0K Nov 21 05:22 .
-rw-------  1 root root  26G Nov 21 05:08 Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf
drwx------ 51 root root 4.0K Nov 21 04:41 ..
-rw-------  1 root root 6.6K Nov 20 20:05 somala_drive_gguf_audit.json
-rw-------  1 root root  27G Nov 14 20:14 qwen3.8-27b-topocbp-Q8_0.gguf
-rw-------  1 root root  11G Nov 12 06:55 deepseek_moe_16b_topocbp_q4_k_m.gguf
-rw-------  1 root root  70G Nov  9 06:01 deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf
-rw-------  1 root root  32G Nov  9 05:57 sarvam-30b-topo-governed-q8_0.gguf
-rw-------  1 root root  16G Nov  9 04:40 muse-glimmer-30b-card-faithful-Q4_K_M.gguf
-rw-------  1 root root  15G Nov  9 04:36 gpt-oss-20b-topo-governed-Q4_K_M.gguf
-rw-------  1 root root  15G Nov  9 04:34 certified-scorpion-topo-qwen-bf16.gguf
-rw-------  1 root root  40G Nov  9 04:31 llama-3.1-70b-topo-governed-q4_k_m.gguf
-rw-------  1 root root 5.0G Nov  9 04:23 nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf
-rw

## hf

In [2]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

LOCAL_GGUF = "/content/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf"
REPO_ID = "frankmorales2020/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0-GGUF"

if not os.path.isfile(LOCAL_GGUF):
    raise FileNotFoundError(f"GGUF not found: {LOCAL_GGUF}")

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)

print(f"[*] Uploading {LOCAL_GGUF} -> https://huggingface.co/{REPO_ID}")
url = api.upload_file(
    path_or_fileobj=LOCAL_GGUF,
    path_in_repo=os.path.basename(LOCAL_GGUF),
    repo_id=REPO_ID,
    repo_type="model",
)
print(f"[+] Uploaded: {url}")

[*] Uploading /content/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf -> https://huggingface.co/frankmorales2020/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0-GGUF


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ca-topo-epoch-1-Q8_0.gguf:   0%|          |  588kB / 26.9GB            

Validating                    :           |   0%            

[+] Uploaded: https://huggingface.co/frankmorales2020/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0-GGUF/commit/4393b1f375e9a2ae5176a798c957d225f3842f93


## inference-hf

In [3]:
import gc
import os
import torch
from google.colab import userdata
from huggingface_hub import hf_hub_download, login
from llama_cpp import Llama

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)

REPO_ID = "frankmorales2020/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0-GGUF"
FILENAME = "Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf"
SEED = 123

PROMPTS = [
    "Explain the difference between synchronous and asynchronous execution in two sentences.",
    "List three practical ways to reduce GPU memory usage during neural network training.",
    "Given a table orders(order_id, customer_id, total_amount, created_at), write a SQL query to find the top 5 customers by total spend.",
]


def format_alpaca(instruction: str) -> str:
    return (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{instruction}\n\n### Response:\n"
    )


model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    token=HF_TOKEN,
)

print(f"[*] Loading GGUF: {model_path}")
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,
    n_ctx=2048,
    flash_attn=True,
    seed=SEED,
    verbose=False,
)
print(f"[+] Loaded GGUF: {model_path}")
print('\n')

for idx, prompt in enumerate(PROMPTS, start=1):
    out = llm(
        format_alpaca(prompt),
        max_tokens=256,
        temperature=0.0,
        stop=["### Instruction:", "\n###"],
        seed=SEED,
        echo=False,
    )
    choice = out["choices"][0]
    text = choice["text"].strip()
    reason = choice.get("finish_reason")
    print(f"--- Prompt {idx} (finish_reason={reason}) ---\nQ: {prompt}\nA: {text}\n")

del llm
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0(…): reconstructing file:   0%|          |  0.00B / 26.9GB            

Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0(…): downloading bytes:           |  0.00B            

[*] Loading GGUF: /root/.cache/huggingface/hub/models--frankmorales2020--Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0-GGUF/snapshots/4393b1f375e9a2ae5176a798c957d225f3842f93/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


[+] Loaded GGUF: /root/.cache/huggingface/hub/models--frankmorales2020--Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0-GGUF/snapshots/4393b1f375e9a2ae5176a798c957d225f3842f93/Gemma-4-26B-A4B-alpaca-topo-epoch-1-Q8_0.gguf


--- Prompt 1 (finish_reason=stop) ---
Q: Explain the difference between synchronous and asynchronous execution in two sentences.
A: Synchronous execution refers to a process where tasks are completed one after another, with each task waiting for the previous one to complete before starting. In contrast, asynchronous execution allows tasks to be started and completed independently, without waiting for other tasks to finish, enabling more efficient use of resources and faster overall execution.

--- Prompt 2 (finish_reason=stop) ---
Q: List three practical ways to reduce GPU memory usage during neural network training.
A: 1. **Model Compression:** One of the most effective ways to reduce GPU memory usage is by compressing the model. This can be done through techniques such a